# Parte 2 – API de lluvias
Grupo 9 · Temporada: 1 de enero al 31 de mayo de 2022

In [1]:
%pip install lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import io
import time
import requests
import pandas as pd

url = "https://es.wikipedia.org/wiki/Departamentos_del_Per%C3%BA"
respuesta = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
tablas = pd.read_html(io.StringIO(respuesta.text))

print("Tablas encontradas:", len(tablas))
for i, t in enumerate(tablas):
    print(i, t.shape, list(t.columns)[:6])

Tablas encontradas: 6
0 (1, 1) ['Departamentos del Perú (con las provincias de régimen especial)']
1 (24, 11) ['Bandera', 'Escudo', 'Departamento', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
2 (8, 5) ['Departamento', 'Capital', 'Creación', 'Desaparición', 'Causa']
3 (2, 11) ['Bandera', 'Escudo', 'Provincia', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
4 (6, 4) ['Provincia', 'Capital', 'Creación', 'Desaparición']
5 (1, 2) [0, 1]


In [3]:
import re

t1 = tablas[1]
t3 = tablas[3]
print("Columnas tabla 1:", list(t1.columns))
print("Columnas tabla 3:", list(t3.columns))

col_cap1 = [c for c in t1.columns if str(c).startswith("Capital")][0]
col_cap3 = [c for c in t3.columns if str(c).startswith("Capital")][0]

deps = t1[["Departamento", col_cap1]].copy()
deps.columns = ["departamento", "capital"]

# Agregamos el Callao desde la tabla 3 (provincias de régimen especial)
callao = t3[t3["Provincia"].astype(str).str.contains("Callao")][["Provincia", col_cap3]].copy()
callao.columns = ["departamento", "capital"]
deps = pd.concat([deps, callao], ignore_index=True)

print("Filas:", len(deps))
print(deps.to_string())

Columnas tabla 1: ['Bandera', 'Escudo', 'Departamento', 'Nombre oficial', 'Abreviatura Nacional', 'Capital', 'Ciudad más poblada', 'Superficie (km²)', 'Población (2025)', 'Creación', 'Ubicación']
Columnas tabla 3: ['Bandera', 'Escudo', 'Provincia', 'Nombre oficial', 'Abreviatura Nacional', 'Capital', 'Ciudad más poblada', 'Superficie (km²)', 'Población (2025)', 'Creación', 'Ubicación']
Filas: 25
      departamento            capital
0         Amazonas        Chachapoyas
1           Áncash             Huaraz
2         Apurímac            Abancay
3         Arequipa           Arequipa
4         Ayacucho           Ayacucho
5        Cajamarca          Cajamarca
6            Cusco              Cusco
7     Huancavelica       Huancavelica
8          Huánuco            Huánuco
9              Ica                Ica
10           Junín           Huancayo
11     La Libertad           Trujillo
12      Lambayeque           Chiclayo
13            Lima  Huacho (de facto)
14          Loreto            I

In [4]:
# quitar notas tipo [nota 6] o [1] y textos entre paréntesis
deps["departamento"] = deps["departamento"].str.replace(r"\[.*?\]", "", regex=True).str.strip()
deps["capital"] = deps["capital"].str.replace(r"\[.*?\]", "", regex=True)
deps["capital"] = deps["capital"].str.replace(r"\(.*?\)", "", regex=True).str.strip()

# decisión: para Lima usamos la ciudad de Lima (ver explicación)
deps.loc[deps["departamento"] == "Lima", "capital"] = "Lima"

print("Filas:", len(deps))
print(deps.to_string())

Filas: 25
     departamento           capital
0        Amazonas       Chachapoyas
1          Áncash            Huaraz
2        Apurímac           Abancay
3        Arequipa          Arequipa
4        Ayacucho          Ayacucho
5       Cajamarca         Cajamarca
6           Cusco             Cusco
7    Huancavelica      Huancavelica
8         Huánuco           Huánuco
9             Ica               Ica
10          Junín          Huancayo
11    La Libertad          Trujillo
12     Lambayeque          Chiclayo
13           Lima              Lima
14         Loreto           Iquitos
15  Madre de Dios  Puerto Maldonado
16       Moquegua          Moquegua
17          Pasco    Cerro de Pasco
18          Piura             Piura
19           Puno              Puno
20     San Martín         Moyobamba
21          Tacna             Tacna
22         Tumbes            Tumbes
23        Ucayali          Pucallpa
24         Callao            Callao


### Tabla de 25 capitales
- La tabla principal de Wikipedia trae **24 departamentos**. El **Callao** no aparece ahí porque es una *provincia constitucional* (régimen especial); lo agregamos desde la tabla de "provincias de régimen especial" de la misma página.
- Limpiamos las notas de Wikipedia (por ejemplo, "Callao[nota 6]").
- **Lima:** Wikipedia pone "Huacho (de facto)", que es la sede del Gobierno Regional de Lima Provincias. Usamos la **ciudad de Lima**, porque el departamento de Lima del INEI (ubigeo 15) incluye Lima Metropolitana y su capital oficial es Lima. Además, "Huacho (de facto)" escrito así no lo encontraría la API.

In [5]:
import unicodedata

def sin_tildes(texto):
    texto = unicodedata.normalize("NFD", str(texto))
    return "".join(c for c in texto if unicodedata.category(c) != "Mn").lower()

filas_geo = []
for _, fila in deps.iterrows():
    r = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                     params={"name": fila["capital"], "count": 10, "language": "es"})
    resultados = r.json().get("results", [])
    # nos quedamos con el PRIMER resultado que esté en el Perú
    elegido = next((x for x in resultados if x.get("country_code") == "PE"), None)
    filas_geo.append({
        "departamento": fila["departamento"],
        "capital": fila["capital"],
        "admin1": elegido.get("admin1") if elegido else None,
        "latitud": elegido["latitude"] if elegido else None,
        "longitud": elegido["longitude"] if elegido else None,
    })
    time.sleep(1)

geo = pd.DataFrame(filas_geo)
# ¿el admin1 que devolvió la API contiene el nombre del departamento? (sin tildes ni mayúsculas)
geo["coincide"] = geo.apply(lambda f: sin_tildes(f["departamento"]) in sin_tildes(f["admin1"]), axis=1)
print(geo.to_string())
print("\nFilas que NO coinciden:", (~geo["coincide"]).sum())

     departamento           capital                               admin1   latitud  longitud  coincide
0        Amazonas       Chachapoyas             Departamento de Amazonas  -6.23169 -77.86903      True
1          Áncash            Huaraz                               Ancash  -9.52614 -77.52869      True
2        Apurímac           Abancay                             Apurímac -13.63426 -72.88380      True
3        Arequipa          Arequipa             Departamento de Arequipa -16.39899 -71.53747      True
4        Ayacucho          Ayacucho             Departamento de Ayacucho -13.16380 -74.22345      True
5       Cajamarca         Cajamarca            Departamento de Cajamarca  -7.16378 -78.50027      True
6           Cusco             Cusco                Departamento de Cusco -13.53188 -71.96701      True
7    Huancavelica      Huancavelica         Departamento de Huancavelica -12.78691 -74.97298      True
8         Huánuco           Huánuco              Departamento de Huánuco 

### ¿Cómo elegimos el resultado correcto de la geocodificación?
La API puede devolver varias ciudades con el mismo nombre (por ejemplo, hay "Lima" y "Trujillo" en otros países). Recorrimos los resultados y nos quedamos con el **primero que tiene `country_code == "PE"`**.
Luego verificamos que `admin1` coincida con nuestro departamento. Como `admin1` viene escrito de distintas formas ("Departamento de Cusco", "Ancash" sin tilde, "Provincia Constitucional del Callao"), comparamos sin tildes ni mayúsculas y revisando si el nombre del departamento está contenido en `admin1`. Las 25 filas coinciden.
En Lima, `admin1` dice "Provincia de Lima", que está dentro del departamento de Lima, así que también es correcto.

In [6]:
filas_lluvia = []
for _, f in geo.iterrows():
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
        "latitude": f["latitud"], "longitude": f["longitud"],
        "start_date": "2022-01-01", "end_date": "2022-05-31",
        "daily": "precipitation_sum", "timezone": "America/Lima",
    })
    valores = r.json()["daily"]["precipitation_sum"]
    validos = [v for v in valores if v is not None]
    filas_lluvia.append({
        "departamento": f["departamento"],
        "capital": f["capital"],
        "latitud": f["latitud"],
        "longitud": f["longitud"],
        "dias": len(valores),
        "dias_sin_dato": len(valores) - len(validos),
        "lluvia_total_mm": round(sum(validos), 1),
        "dias_lluvia_fuerte": sum(1 for v in validos if v >= 20),
    })
    time.sleep(1)   # pausa entre pedidos

lluvia = pd.DataFrame(filas_lluvia)
print("Total de días sin dato (None):", lluvia["dias_sin_dato"].sum())
print(lluvia.sort_values("lluvia_total_mm", ascending=False).to_string())

Total de días sin dato (None): 0
     departamento           capital   latitud  longitud  dias  dias_sin_dato  lluvia_total_mm  dias_lluvia_fuerte
14         Loreto           Iquitos  -3.74814 -73.25290   151              0           1581.8                  25
23        Ucayali          Pucallpa  -8.37915 -74.55387   151              0           1288.8                  20
1          Áncash            Huaraz  -9.52614 -77.52869   151              0            975.3                   7
10          Junín          Huancayo -12.06866 -75.21027   151              0            910.4                   5
15  Madre de Dios  Puerto Maldonado -12.58946 -69.19948   151              0            908.0                  13
7    Huancavelica      Huancavelica -12.78691 -74.97298   151              0            803.8                   1
20     San Martín         Moyobamba  -6.03441 -76.97423   151              0            789.8                   4
19           Puno              Puno -15.84003 -70.02198

### Días sin dato
Contamos cuántos días llegaron con `None` en `precipitation_sum`: **0 en los 25 departamentos**. Todos tienen los 151 días de la temporada (1 de enero al 31 de mayo de 2022). Igual dejamos el código preparado: si hubiera `None`, esos días se excluyen de la suma y se reportan en la columna `dias_sin_dato`, en lugar de contarlos como 0 mm.

In [7]:
lluvia[["departamento", "capital", "latitud", "longitud", "lluvia_total_mm", "dias_lluvia_fuerte"]].to_csv(
    "datos/lluvias_por_departamento.csv", index=False, encoding="utf-8-sig")
print("Guardado:", len(lluvia), "filas")

Guardado: 25 filas
